# Extract Boundary Tokens - Simple Version
Based on camelbert_inference_for_export.ipynb

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install transformers torch tqdm -q
print("Dependencies OK")

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load model from Drive
model_path = Path('checkpoints/camelbert_binary_classification_final')
print(f"Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("Model loaded")

In [ ]:
# Load corpus
corpus_file = Path('data/processed/kitab_uqala_reference_corpus.txt')
print(f"Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    text = f.read()
print(f"Corpus: {len(text):,} chars")

In [ ]:
# Run inference with offset mapping
print("Running inference...")
encoded = tokenizer(
    text,
    max_length=512,
    padding='max_length',
    truncation=True,
    return_offsets_mapping=True,
    return_tensors='pt'
)

with torch.no_grad():
    if torch.cuda.is_available():
        outputs = model(encoded['input_ids'].cuda(), attention_mask=encoded['attention_mask'].cuda())
    else:
        outputs = model(**encoded)
    logits = outputs.logits[0]

predictions = np.argmax(logits.cpu().numpy(), axis=-1)
tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
offsets = encoded['offset_mapping'][0].numpy()

print(f"Inference complete")
print(f"  Total tokens: {len(tokens):,}")
print(f"  Boundary tokens: {(predictions == 1).sum():,}")

In [ ]:
# Extract boundary tokens
print("Extracting boundary tokens...")
boundary_tokens = []
boundary_indices = []

for idx, (token, pred) in enumerate(zip(tokens, predictions)):
    if pred == 1:  # Boundary token
        boundary_tokens.append(token)
        boundary_indices.append(idx)

print(f"Extracted: {len(boundary_tokens):,} boundary tokens")
print(f"Percentage: {100 * len(boundary_tokens) / len(tokens):.2f}%")

In [ ]:
# Preview
print("\nFirst 40 boundary tokens:")
for i, token in enumerate(boundary_tokens[:40], 1):
    print(f"{i:3d}. {token}")

In [ ]:
# Save to JSON
print("\nSaving to JSON...")
results = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus.txt',
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(tokens),
        'model': 'camelbert_binary_classification_final',
        'boundary_tokens_count': len(boundary_tokens),
        'boundary_percentage': round(100 * len(boundary_tokens) / len(tokens), 2),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
}

output_file = Path('results/camelbert_boundary_tokens_clean.json')
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

file_size = output_file.stat().st_size / 1024
print(f"Saved: {output_file}")
print(f"Size: {file_size:.1f} KB")
print(f"\nBoundary tokens: {len(boundary_tokens):,}")
print(f"Done!")